# ColabCluster: register and monitor one Colab worker
Run cells in order. Select a GPU runtime in Colab if available; CPU-only operation also works.
Start the controller on your PC first, then manually configure a reachable endpoint forwarding to port 8000.
**127.0.0.1 in Colab refers to Colab, not your PC.** No tunnel is created by this notebook.
The development controller has no authentication: keep any temporary exposure controlled and close it after testing.
Publish the Step 6 files to your repository before cloning; local uncommitted changes are not available in Colab.


## Alternative: upload the prepared Step 6 bundle
If your repository has not been updated, upload `colabcluster-worker.zip` through the Colab Files sidebar.
Skip the clone/setup cell and run the following in a new Python cell, then continue with configuration and hardware verification:
```python
import sys, subprocess, zipfile
from pathlib import Path
project = Path("/content/colabcluster-step6")
with zipfile.ZipFile("/content/colabcluster-worker.zip") as bundle:
    bundle.extractall(project)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(project / "requirements.txt")], check=True)
```
Restart the old worker before launching the updated entry point. The archive contains project source only, with no credentials or environment files.


In [ ]:
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/MUKILAN0608/ColabCluster.git"
checkout = Path("/content/ColabCluster")
if not checkout.exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(checkout)], check=True)
# Support both the repository-root and nested workspace layouts.
project = next((p for p in (checkout, checkout / "colabcluster")
                if (p / "worker" / "heartbeat.py").is_file()), None)
if project is None:
    raise RuntimeError("This checkout lacks Step 6. Publish the new files and use an updated checkout.")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(project / "requirements.txt")], check=True)


In [ ]:
import os

CONTROLLER_URL = "https://YOUR-REACHABLE-CONTROLLER-ENDPOINT"
if "YOUR-REACHABLE" in CONTROLLER_URL:
    raise ValueError("Replace CONTROLLER_URL with your manually configured reachable endpoint.")
os.environ["COLABCLUSTER_CONTROLLER_URL"] = CONTROLLER_URL
os.environ["COLABCLUSTER_WORKER_ID"] = "COLAB-01"

os.environ["COLABCLUSTER_HEARTBEAT_INTERVAL"] = "10"


## Verify hardware
This uses the same detector as the worker. PyTorch is optional and is not installed by this project.
If CUDA is usable, its first device name and VRAM are shown. Memory units are GiB.


In [ ]:
subprocess.run([sys.executable, "-m", "worker.worker"], cwd=project, check=True)


## Optional Step 6.6: remote GPU diagnostic
Use the updated `colabcluster-worker.zip` source bundle (extract it to the same project directory).
Before starting the worker, manually create a reachable endpoint forwarding to Colab port 8001.
This is a second tunnel: the PC tunnel forwards to the controller, while this endpoint forwards to Colab.
If cloudflared is installed in Colab, run it in a background subprocess with logs, as shown in the README Remote GPU Execution Test section.
Set these values in a new Python cell before starting the worker:
```python
os.environ["COLABCLUSTER_WORKER_URL"] = "https://YOUR-COLAB-WORKER-TUNNEL"
os.environ["COLABCLUSTER_WORKER_ID"] = "COLAB-GPU-TEST"
```
Then start the worker below. The runtime starts `/gpu-test` on port 8001 and keeps heartbeats running.
From the PC project directory run `python testing/gpu_test.py --worker-id COLAB-GPU-TEST`.
Do not set WORKER_URL to the PC tunnel. Old manual heartbeat cells do not implement this endpoint.


## Register and send heartbeats
This cell registers and sends periodic heartbeats; it executes no tasks.
Interrupt normally to attempt unregister. For the offline test, set CRASH_TEST_ON_INTERRUPT=True
before starting this cell; interruption then kills the process without unregistering.
Wait over 35 seconds with defaults and verify the record disappears from /workers.
After expiry, rerun registration with the same ID before resuming heartbeats.
On your PC, verify `GET http://127.0.0.1:8000/workers` contains `COLAB-01` and this runtime's hardware.
Verify last_seen advances across two queries at least 10 seconds apart. Unknown-worker errors do not auto-register.


In [ ]:
CRASH_TEST_ON_INTERRUPT = False  # Set True only for the offline detection test.
process = subprocess.Popen([sys.executable, "-u", "-m", "worker.colab_worker"], cwd=project)
try:
    result = process.wait()
    if result:
        raise RuntimeError(f"Worker exited with status {result}; inspect the error above.")
except KeyboardInterrupt:
    if CRASH_TEST_ON_INTERRUPT:
        process.kill()
    else:
        process.terminate()
    try:
        process.wait(timeout=25)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    print("Worker process stopped.")
